# v81 Gemini 3.7 vs 3.8 pairwise specialist — isolated OOF diagnostic

This experiment does not modify v81 and does not submit. It uses the v81 clean-pool, four-view SVC backbone with 5-fold `GroupKFold` by `prompt_id`. The only allowed change is a rerank between `gemini-3.7-flash` and `gemini-3.8-flash` when the v81 backbone already predicts one of those two labels. The specialist is trained on clean external-pool rows only, matching the v81 sibling-rerank contract. Promotion requires a predeclared +0.010 balanced-accuracy gain.

In [ ]:
import glob, json, time, re
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score, recall_score

LABELS = ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
          'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
          'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5',
          'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']
LABEL2I = {x:i for i,x in enumerate(LABELS)}
PAIR = ['gemini-3.7-flash', 'gemini-3.8-flash']
PAIR_IDX = np.array([LABEL2I[x] for x in PAIR])
THRESHOLDS = [0.00, 0.10, 0.20, 0.35, 0.50, 0.70]
PROMOTION_MARGIN = 0.010

paths = sorted(glob.glob('/kaggle/input/**/train.csv', recursive=True))
assert paths, 'train.csv not found'
train = pd.read_csv(paths[0])
assert {'id','prompt_id','text','model'} <= set(train.columns)
train['text'] = train['text'].fillna('').astype(str)
assert sorted(train.model.unique()) == sorted(LABELS)
y = train.model.map(LABEL2I).to_numpy()
groups = train.prompt_id.astype(str).to_numpy()

def load_pool():
    candidates = [('pool_own_v3.parquet','gtuli12 cleanest'),('pool_own_v2.parquet','gtuli12 v2'),
                  ('ntl_open_pool.parquet','open pool clean'),('merged_corpus.csv','merged fallback'),('pool_v4.parquet','pool v4')]
    train_norm = set(train.text.str.replace(r'\s+',' ',regex=True).str.strip())
    for pattern,name in candidates:
        hits = sorted(glob.glob('/kaggle/input/**/'+pattern, recursive=True))
        if not hits: continue
        try:
            df = pd.read_parquet(hits[0]) if hits[0].endswith('.parquet') else pd.read_csv(hits[0])
        except Exception: continue
        if not {'text','model'} <= set(df.columns): continue
        df = df.dropna(subset=['text','model']).copy()
        df.text = df.text.astype(str); df.model = df.model.astype(str)
        df = df[df.model.isin(LABELS)]
        df = df[df.text.str.split().str.len().between(30,300)]
        norm = df.text.str.replace(r'\s+',' ',regex=True).str.strip()
        df = df[~norm.isin(train_norm)].copy()
        df = df.loc[~norm[~norm.isin(train_norm)].duplicated()].reset_index(drop=True)
        print({'pool_source':name,'rows':len(df)}, flush=True)
        return df[['text','model']]
    raise FileNotFoundError('clean v81 pool not found')
pool = load_pool()
print({'train_rows':len(train),'prompt_groups':len(np.unique(groups)),'pool_rows':len(pool),'pair_counts':pool.model.value_counts().reindex(PAIR).to_dict()}, flush=True)

In [ ]:
SMALL_WORDS = set('a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little'.split())
def shape_text(t): return re.sub(r'[^\W\d_]+', lambda m:'X' if m[0].isupper() else 'x', t)
def small_words_text(t):
    return ' '.join(w.lower() if w.lower() in SMALL_WORDS or not w.isalnum() else 'CONTENT' for w in re.findall(r'\w+|[^\w\s]', t))
def make_features(fit_texts, va_texts):
    A=[]; B=[]
    specs=[('char',3,5,100000,1.0),('word',1,3,80000,0.7),('shape',2,6,30000,0.5),('small',1,3,30000,0.5)]
    for view,lo,hi,mx,w in specs:
        prep = shape_text if view=='shape' else small_words_text if view=='small' else (lambda s:s)
        analyzer = 'char' if view!='small' else 'word'
        opts=dict(analyzer=analyzer,ngram_range=(lo,hi),lowercase=False,sublinear_tf=True,min_df=2,max_features=mx,dtype=np.float32)
        if analyzer=='word': opts['token_pattern']=r'(?u)\b\w+\b|[^\w\s]'
        vec=TfidfVectorizer(**opts)
        A.append(vec.fit_transform([prep(t) for t in fit_texts])*w)
        B.append(vec.transform([prep(t) for t in va_texts])*w)
    return hstack(A,format='csr'), hstack(B,format='csr')

In [ ]:
fold_records=[]
oof_base=np.full(len(train),-1,dtype=int)
oof_pair={t:np.full(len(train),-1,dtype=int) for t in THRESHOLDS}
t0=time.time()
gkf=GroupKFold(n_splits=5)
for fold,(itr,iva) in enumerate(gkf.split(train,y,groups),1):
    assert set(groups[itr]).isdisjoint(set(groups[iva]))
    fit_texts=train.text.iloc[itr].tolist()+pool.text.tolist()
    fit_y=np.r_[y[itr], pool.model.map(LABEL2I).to_numpy()]
    Xtr,Xva=make_features(fit_texts,train.text.iloc[iva].tolist())
    base=LinearSVC(C=0.5,class_weight='balanced',max_iter=5000,random_state=100+fold)
    base.fit(Xtr,fit_y)
    dec=base.decision_function(Xva)
    base_pred=dec.argmax(1)
    order=np.argsort(dec,axis=1); top=order[:,-1]; second=order[:,-2]
    margin=dec[np.arange(len(iva)),top]-dec[np.arange(len(iva)),second]
    pair_pool=pool[pool.model.isin(PAIR)]
    specialist=LinearSVC(C=1.0,class_weight='balanced',max_iter=5000,random_state=200+fold)
    specialist.fit(Xtr[len(itr):][pair_pool.index.to_numpy()], pair_pool.model.map(LABEL2I).to_numpy())
    spec_pred=specialist.predict(Xva)
    oof_base[iva]=base_pred
    rec={'fold':fold,'rows':int(len(iva)),'base_bal_acc':float(balanced_accuracy_score(y[iva],base_pred)),
         'pair_gate_rows':int(np.isin(top,PAIR_IDX).sum())}
    for t in THRESHOLDS:
        gate=np.isin(top,PAIR_IDX) & ((margin <= t) if t>0 else np.zeros(len(iva),dtype=bool))
        pred=base_pred.copy(); pred[gate]=spec_pred[gate]
        oof_pair[t][iva]=pred
        rec[f't_{t:.2f}']={'balanced_accuracy':float(balanced_accuracy_score(y[iva],pred)),
                           'changed_rows':int((pred!=base_pred).sum()),'gate_rows':int(gate.sum())}
    fold_records.append(rec); print(rec,flush=True)
summary=[]
base_score=balanced_accuracy_score(y,oof_base)
for t in THRESHOLDS:
    pred=oof_pair[t]
    recalls=recall_score(y,pred,labels=np.arange(len(LABELS)),average=None,zero_division=0)
    pair_macro=float(np.mean([recalls[LABEL2I[x]] for x in PAIR]))
    summary.append({'threshold':t,'balanced_accuracy':float(balanced_accuracy_score(y,pred)),
                    'delta_vs_no_specialist':float(balanced_accuracy_score(y,pred)-base_score),
                    'pair_macro_recall':pair_macro,'changed_rows':int((pred!=oof_base).sum()),
                    'per_class_recall':{LABELS[i]:float(recalls[i]) for i in range(len(LABELS))}})
positive=[r for r in summary if r['delta_vs_no_specialist']>=PROMOTION_MARGIN]
selected=max(positive,key=lambda r:r['balanced_accuracy']) if positive else None
result={'experiment':'v81_gemini_3.7_vs_3.8_pairwise','protocol':'5-fold GroupKFold by prompt_id; v81 four-view SVC + clean pool backbone','rows':int(len(train)),'prompt_groups':int(len(np.unique(groups))),
        'pool_rows':int(len(pool)),'pair':PAIR,'base_balanced_accuracy':float(base_score),'predeclared_promotion_margin':PROMOTION_MARGIN,
        'thresholds':summary,'selected':selected,'folds':fold_records,'elapsed_seconds':round(time.time()-t0,1),
        'note':'OOF diagnostic uses the v81 SVC backbone; full GPU members are not rerun in each fold. No hidden labels and no submission.'}
Path('/kaggle/working/gemini_pairwise_results.json').write_text(json.dumps(result,indent=2))
print(json.dumps({k:result[k] for k in ['experiment','protocol','base_balanced_accuracy','selected','elapsed_seconds']},indent=2),flush=True)